# IPL Historical Weather Enrichment

This notebook prepares IPL fixture data, enriches match records with historical weather observations from Open-Meteo, and explores weather-based dew-likelihood patterns alongside toss decisions and match outcomes.

## Before running
- Place `IPL_matches_enriched_step2.csv` in `data/input/`.
- Run the cells from top to bottom.
- The notebook writes intermediate files to `data/intermediate/`, final datasets and summaries to `data/output/`, and reusable API caches to `data/cache/`.
- Install the required packages first: `pandas`, `numpy`, and `requests`.
- The notebook needs an internet connection to retrieve historical weather data.

**Important:** Dew likelihood is a weather-based proxy, not a direct measurement of dew on the pitch.


## 1. Repair fixture date and time fields
Reads the raw Step 2 fixture file and writes a normalized intermediate CSV.

In [ ]:
from pathlib import Path
import pandas as pd
import os

# Resolve the repository root whether this notebook is launched from
# the repository root or from its notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "data" / "input"
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"

for directory in (INTERMEDIATE_DIR, OUTPUT_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# ============================================================
# REPAIR STEP 2 DATE / TIME FORMATTING
# ============================================================

INPUT_FILE = INPUT_DIR / "IPL_matches_enriched_step2.csv"

OUTPUT_FILE = INTERMEDIATE_DIR / "IPL_matches_enriched_step2_fixed.csv"

print("=" * 70)
print("REPAIRING IPL STEP 2 DATE/TIME DATA")
print("=" * 70)

# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("\nOriginal shape:", df.shape)

# ------------------------------------------------------------
# IMPORTANT:
# The CSV stores dates as DD-MM-YYYY.
# Explicitly tell pandas that.
# ------------------------------------------------------------

df["date"] = pd.to_datetime(
    df["date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# ------------------------------------------------------------
# Parse scheduled match time
# Format:
# DD-MM-YYYY HH:MM
# ------------------------------------------------------------

df["match_start_time"] = pd.to_datetime(
    df["match_start_time"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

# ------------------------------------------------------------
# Re-create match timing
# ------------------------------------------------------------

df["match_timing"] = df["match_start_time"].apply(
    lambda x:
        "Day"
        if pd.notna(x) and x.hour < 17
        else "Night"
        if pd.notna(x)
        else pd.NA
)

# ------------------------------------------------------------
# Re-create match start hour
# ------------------------------------------------------------

df["match_start_hour"] = (
    df["match_start_time"].dt.hour
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION")
print("=" * 70)

print(
    "Total matches:",
    len(df)
)

print(
    "Valid dates:",
    df["date"].notna().sum()
)

print(
    "Scheduled times:",
    df["match_start_time"].notna().sum()
)

print(
    "Missing scheduled times:",
    df["match_start_time"].isna().sum()
)

print("\nMatch timing:")
print(
    df["match_timing"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# Show sample
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE")
print("=" * 70)

print(
    df[
        [
            "match_id",
            "date",
            "season",
            "team1",
            "team2",
            "match_start_time",
            "match_timing"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("FIXED FILE SAVED")
print("=" * 70)

print(OUTPUT_FILE)

print("\nShape:", df.shape)

## 2. Validate the repaired fixture data
Checks date/time parsing and displays sample records before weather enrichment.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_FILE = PROJECT_ROOT / "data" / "intermediate" / "IPL_matches_enriched_step2_fixed.csv"


file = INPUT_FILE

df = pd.read_csv(file)

df["date"] = pd.to_datetime(
    df["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df["match_start_time"] = pd.to_datetime(
    df["match_start_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print("Total matches:", len(df))
print("Scheduled times:", df["match_start_time"].notna().sum())
print("Missing times:", df["match_start_time"].isna().sum())

print("\nFirst 10 matches:")
print(
    df[
        [
            "match_id",
            "date",
            "team1",
            "team2",
            "match_start_time",
            "match_timing"
        ]
    ].head(10).to_string(index=False)
)

## 3. Enrich matches with historical weather and analyze dew likelihood
Retrieves weather observations, calculates proxy variables, and exports the enriched dataset and summary tables.

In [ ]:
# ==============================================================
# IPL HISTORICAL WEATHER ENRICHMENT — FINAL VERSION
# ==============================================================
#
# Input:
#   IPL_matches_enriched_step2_fixed.csv
#
# Output:
#   IPL_matches_enriched_step3_weather.csv
#   weather_by_timing.csv
#   dew_toss_analysis.csv
#   dew_outcome_analysis.csv
#   dew_timing_toss_analysis.csv
#
# Weather source:
#   Open-Meteo Historical Weather / ERA5
#
# IMPORTANT:
#   Dew variables are an analytical proxy, NOT measured pitch dew.
# ==============================================================

from pathlib import Path
import os
import time
import json
import requests
import numpy as np
import pandas as pd


# ==============================================================
# 1. FILE PATHS
# ==============================================================

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "data" / "input"
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"

for directory in (INTERMEDIATE_DIR, OUTPUT_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

INPUT_FILE = INTERMEDIATE_DIR / "IPL_matches_enriched_step2_fixed.csv"

OUTPUT_FILE = OUTPUT_DIR / "IPL_matches_enriched_step3_weather.csv"

WEATHER_CACHE_FILE = CACHE_DIR / "ipl_weather_cache.json"

COORD_CACHE_FILE = CACHE_DIR / "ipl_city_coordinates.json"


# ==============================================================
# 2. SETTINGS
# ==============================================================

WEATHER_API = "https://archive-api.open-meteo.com/v1/archive"
GEOCODING_API = "https://geocoding-api.open-meteo.com/v1/search"

TIMEZONE = "auto"

# Weather variables
HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "wind_speed_10m",
]

# Weather request retry settings
MAX_RETRIES = 3
RETRY_DELAY = 3


# ==============================================================
# 3. LOAD DATA
# ==============================================================

print("=" * 70)
print("IPL HISTORICAL WEATHER ENRICHMENT — FINAL VERSION")
print("=" * 70)

print("\nReading:")
print(INPUT_FILE)

df = pd.read_csv(INPUT_FILE)

print("\nInput shape:", df.shape)


# ==============================================================
# 4. CORRECT DATE / TIME PARSING
# ==============================================================

df["date"] = pd.to_datetime(
    df["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df["match_start_time"] = pd.to_datetime(
    df["match_start_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Recreate timing from actual scheduled hour
df["match_timing"] = df["match_start_time"].apply(
    lambda x:
        "Day"
        if pd.notna(x) and x.hour < 17
        else "Night"
        if pd.notna(x)
        else pd.NA
)

df["match_start_hour"] = df["match_start_time"].dt.hour


print("\nDate/time validation:")
print("Total matches:", len(df))
print("Valid dates:", df["date"].notna().sum())
print("Scheduled times:", df["match_start_time"].notna().sum())
print("Missing scheduled times:", df["match_start_time"].isna().sum())


# ==============================================================
# 5. NORMALIZE WEATHER CITY NAMES
# ==============================================================
#
# IMPORTANT:
# Original IPL city column is preserved.
# weather_city is used only for weather lookup.
# ==============================================================

CITY_ALIASES = {
    "Bangalore": "Bengaluru",
    "Bengaluru": "Bengaluru",

    # Chandigarh-region IPL venues
    "New Chandigarh": "New Chandigarh",
    "Mohali": "Mohali",
    "Chandigarh": "Chandigarh",
}

df["weather_city"] = df["city"].replace(CITY_ALIASES)


print("\nOriginal cities:", df["city"].nunique())
print("Weather lookup cities:", df["weather_city"].nunique())


# ==============================================================
# 6. MANUAL IPL CITY COORDINATES
# ==============================================================
#
# These are used for known IPL cities to avoid geocoder mistakes.
#
# Particularly important:
# Bangalore was previously incorrectly geocoded to Pakistan.
#
# New Chandigarh is manually specified because geocoding failed.
# ==============================================================

MANUAL_COORDINATES = {

    "Abu Dhabi": (24.4512, 54.3970),

    "Ahmedabad": (23.0258, 72.5873),

    "Bengaluru": (12.9719, 77.5937),

    "Chandigarh": (30.7363, 76.7884),

    "Chennai": (13.0878, 80.2785),

    "Delhi": (28.6519, 77.2315),

    "Dharamsala": (32.2201, 76.3201),

    "Dubai": (25.0772, 55.3093),

    "Guwahati": (26.1844, 91.7458),

    "Hyderabad": (17.3840, 78.4564),

    "Indore": (22.7179, 75.8333),

    "Jaipur": (26.9196, 75.7878),

    "Kanpur": (26.4652, 80.3498),

    "Kolkata": (22.5626, 88.3630),

    "Lucknow": (26.8393, 80.9231),

    "Mohali": (30.6800, 76.7221),

    "Mumbai": (19.0728, 72.8826),

    "Navi Mumbai": (19.0368, 73.0158),

    "New Chandigarh": (30.7290, 76.8530),

    "Pune": (18.5196, 73.8554),

    "Raipur": (21.2333, 81.6333),

    "Rajkot": (22.2916, 70.7932),

    "Sharjah": (25.3342, 55.4122),

    "Visakhapatnam": (17.6801, 83.2016),
}


# ==============================================================
# 7. LOAD COORDINATE CACHE
# ==============================================================

if os.path.exists(COORD_CACHE_FILE):

    try:
        with open(COORD_CACHE_FILE, "r", encoding="utf-8") as f:
            coordinate_cache = json.load(f)

        print("\nExisting coordinate cache loaded.")

    except Exception:
        coordinate_cache = {}

else:
    coordinate_cache = {}


# ==============================================================
# 8. FORCE CORRECT MANUAL COORDINATES
# ==============================================================

for city, coords in MANUAL_COORDINATES.items():

    coordinate_cache[city] = {
        "latitude": coords[0],
        "longitude": coords[1],
    }


# Save corrected coordinate cache

with open(COORD_CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(coordinate_cache, f, indent=2)


print("\nCorrected city coordinates loaded.")


# ==============================================================
# 9. DISPLAY COORDINATES
# ==============================================================

print("\n" + "=" * 70)
print("CITY COORDINATES")
print("=" * 70)

for city in sorted(coordinate_cache.keys()):

    coords = coordinate_cache[city]

    print(
        f"{city:<35} "
        f"{coords['latitude']:.4f}, "
        f"{coords['longitude']:.4f}"
    )


# ==============================================================
# 10. LOAD WEATHER CACHE
# ==============================================================

if os.path.exists(WEATHER_CACHE_FILE):

    try:

        with open(WEATHER_CACHE_FILE, "r", encoding="utf-8") as f:
            weather_cache = json.load(f)

        print("\nExisting weather cache loaded.")

    except Exception:

        weather_cache = {}

else:

    weather_cache = {}


# ==============================================================
# 11. WEATHER DOWNLOAD FUNCTION
# ==============================================================

def download_weather(
    latitude,
    longitude,
    start_date,
    end_date
):

    params = {
        "latitude": latitude,
        "longitude": longitude,

        "start_date": start_date,
        "end_date": end_date,

        "hourly": ",".join(HOURLY_VARIABLES),

        "timezone": TIMEZONE,
    }

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = requests.get(
                WEATHER_API,
                params=params,
                timeout=60
            )

            response.raise_for_status()

            return response.json()

        except Exception as e:

            print(
                f"  Weather attempt {attempt}/{MAX_RETRIES} failed: {e}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

    return None


# ==============================================================
# 12. PREPARE MATCHES REQUIRING WEATHER
# ==============================================================

weather_matches = df[
    df["match_start_time"].notna()
].copy()

print("\n" + "=" * 70)
print("WEATHER REQUEST PREPARATION")
print("=" * 70)

print("Matches requiring weather:", len(weather_matches))


# ==============================================================
# 13. CITY-YEAR GROUPS
# ==============================================================

weather_matches["weather_year"] = (
    weather_matches["match_start_time"].dt.year
)

city_year_groups = (
    weather_matches
    .groupby(["weather_city", "weather_year"])
)

print(
    "City-year groups:",
    len(city_year_groups)
)


# ==============================================================
# 14. DOWNLOAD WEATHER
# ==============================================================

weather_records = {}

print("\n" + "=" * 70)
print("WEATHER DOWNLOAD")
print("=" * 70)


for i, ((city, year), group) in enumerate(
    city_year_groups,
    start=1
):

    print(
        f"\n[{i}/{len(city_year_groups)}] "
        f"{city} - {year}"
    )

    if city not in coordinate_cache:

        print("  Coordinates unavailable.")

        continue

    coords = coordinate_cache[city]

    latitude = coords["latitude"]
    longitude = coords["longitude"]

    dates = group["match_start_time"]

    start_date = dates.min().strftime("%Y-%m-%d")
    end_date = dates.max().strftime("%Y-%m-%d")

    print(
        f"  Period: {start_date} → {end_date}"
    )

    cache_key = (
        f"{city}|{year}|"
        f"{start_date}|{end_date}"
    )

    # ----------------------------------------------------------
    # USE CACHE IF AVAILABLE
    # ----------------------------------------------------------

    if cache_key in weather_cache:

        weather_data = weather_cache[cache_key]

        print("  Loaded from cache.")

    else:

        weather_data = download_weather(
            latitude,
            longitude,
            start_date,
            end_date
        )

        if weather_data is None:

            print("  Weather download failed.")

            continue

        weather_cache[cache_key] = weather_data

        with open(
            WEATHER_CACHE_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                weather_cache,
                f
            )

        # Small delay
        time.sleep(0.5)

    # ----------------------------------------------------------
    # VALIDATE WEATHER DATA
    # ----------------------------------------------------------

    if (
        not weather_data
        or "hourly" not in weather_data
    ):

        print("  Invalid weather response.")

        continue

    hourly = weather_data["hourly"]

    times = hourly.get("time", [])

    if not times:

        print("  No hourly records.")

        continue

    records = []

    for j, timestamp in enumerate(times):

        record = {
            "timestamp": timestamp
        }

        for variable in HOURLY_VARIABLES:

            values = hourly.get(variable, [])

            if j < len(values):

                record[variable] = values[j]

            else:

                record[variable] = np.nan

        records.append(record)

    weather_records[cache_key] = pd.DataFrame(records)

    print(
        "  Retrieved",
        len(records),
        "hourly records."
    )


# ==============================================================
# 15. MATCH WEATHER TO EACH IPL MATCH
# ==============================================================

print("\n" + "=" * 70)
print("MATCHING WEATHER TO IPL MATCHES")
print("=" * 70)


def get_weather_for_datetime(
    city,
    timestamp,
    offset_hours=0
):

    if pd.isna(timestamp):
        return None

    lookup_time = (
        timestamp
        + pd.Timedelta(hours=offset_hours)
    )

    year = lookup_time.year

    # Find matching cached city-year weather data
    possible_keys = [
        key
        for key in weather_records.keys()
        if key.startswith(f"{city}|{year}|")
    ]

    if not possible_keys:
        return None

    weather_df = weather_records[possible_keys[0]]

    weather_df = weather_df.copy()

    weather_df["timestamp"] = pd.to_datetime(
        weather_df["timestamp"],
        errors="coerce"
    )

    match = weather_df[
        weather_df["timestamp"] == lookup_time
    ]

    if match.empty:

        # Try nearest available hour
        weather_df["time_diff"] = abs(
            weather_df["timestamp"] - lookup_time
        )

        nearest = weather_df[
            weather_df["time_diff"]
            <= pd.Timedelta(hours=1)
        ].sort_values("time_diff")

        if nearest.empty:
            return None

        row = nearest.iloc[0]

    else:

        row = match.iloc[0]

    return {
        "temperature_2m": row["temperature_2m"],
        "relative_humidity_2m": row[
            "relative_humidity_2m"
        ],
        "dew_point_2m": row[
            "dew_point_2m"
        ],
        "precipitation": row[
            "precipitation"
        ],
        "wind_speed_10m": row[
            "wind_speed_10m"
        ],
    }


# ==============================================================
# 16. CREATE WEATHER COLUMNS
# ==============================================================

weather_columns = [

    "weather_start_temperature_2m",
    "weather_2h_temperature_2m",

    "weather_start_relative_humidity_2m",
    "weather_2h_relative_humidity_2m",

    "weather_start_dew_point_2m",
    "weather_2h_dew_point_2m",

    "weather_start_precipitation",
    "weather_2h_precipitation",

    "weather_start_wind_speed_10m",
    "weather_2h_wind_speed_10m",

    "dew_point_depression_start",
    "dew_point_depression_2h",

    "dew_proxy_score",
    "dew_likelihood",
]


for col in weather_columns:

    if col == "dew_likelihood":

        df[col] = pd.NA

    else:

        df[col] = np.nan


# ==============================================================
# 17. WEATHER MATCHING LOOP
# ==============================================================

weather_start_matches = 0
weather_2h_matches = 0


for idx, row in df.iterrows():

    timestamp = row["match_start_time"]

    city = row["weather_city"]

    # ----------------------------------------------------------
    # START TIME WEATHER
    # ----------------------------------------------------------

    start_weather = get_weather_for_datetime(
        city,
        timestamp,
        offset_hours=0
    )

    # ----------------------------------------------------------
    # +2 HOUR WEATHER
    # ----------------------------------------------------------

    weather_2h = get_weather_for_datetime(
        city,
        timestamp,
        offset_hours=2
    )

    # ----------------------------------------------------------
    # START WEATHER
    # ----------------------------------------------------------

    if start_weather is not None:

        weather_start_matches += 1

        df.at[
            idx,
            "weather_start_temperature_2m"
        ] = start_weather["temperature_2m"]

        df.at[
            idx,
            "weather_start_relative_humidity_2m"
        ] = start_weather[
            "relative_humidity_2m"
        ]

        df.at[
            idx,
            "weather_start_dew_point_2m"
        ] = start_weather[
            "dew_point_2m"
        ]

        df.at[
            idx,
            "weather_start_precipitation"
        ] = start_weather[
            "precipitation"
        ]

        df.at[
            idx,
            "weather_start_wind_speed_10m"
        ] = start_weather[
            "wind_speed_10m"
        ]

    # ----------------------------------------------------------
    # +2 HOURS WEATHER
    # ----------------------------------------------------------

    if weather_2h is not None:

        weather_2h_matches += 1

        df.at[
            idx,
            "weather_2h_temperature_2m"
        ] = weather_2h["temperature_2m"]

        df.at[
            idx,
            "weather_2h_relative_humidity_2m"
        ] = weather_2h[
            "relative_humidity_2m"
        ]

        df.at[
            idx,
            "weather_2h_dew_point_2m"
        ] = weather_2h[
            "dew_point_2m"
        ]

        df.at[
            idx,
            "weather_2h_precipitation"
        ] = weather_2h[
            "precipitation"
        ]

        df.at[
            idx,
            "weather_2h_wind_speed_10m"
        ] = weather_2h[
            "wind_speed_10m"
        ]

    # ----------------------------------------------------------
    # DEW POINT DEPRESSION
    # ----------------------------------------------------------

    start_temp = df.at[
        idx,
        "weather_start_temperature_2m"
    ]

    start_dew = df.at[
        idx,
        "weather_start_dew_point_2m"
    ]

    temp_2h = df.at[
        idx,
        "weather_2h_temperature_2m"
    ]

    dew_2h = df.at[
        idx,
        "weather_2h_dew_point_2m"
    ]

    if (
        pd.notna(start_temp)
        and pd.notna(start_dew)
    ):

        df.at[
            idx,
            "dew_point_depression_start"
        ] = start_temp - start_dew

    if (
        pd.notna(temp_2h)
        and pd.notna(dew_2h)
    ):

        df.at[
            idx,
            "dew_point_depression_2h"
        ] = temp_2h - dew_2h

    # ----------------------------------------------------------
    # DEW PROXY SCORE
    # ----------------------------------------------------------

    humidity = df.at[
        idx,
        "weather_2h_relative_humidity_2m"
    ]

    depression = df.at[
        idx,
        "dew_point_depression_2h"
    ]

    if (
        pd.notna(humidity)
        and pd.notna(depression)
    ):

        humidity_score = np.clip(
            (humidity - 60) / 40,
            0,
            1
        )

        depression_score = np.clip(
            (8 - depression) / 8,
            0,
            1
        )

        dew_score = (
            0.60 * humidity_score
            + 0.40 * depression_score
        ) * 100

        df.at[
            idx,
            "dew_proxy_score"
        ] = round(
            dew_score,
            2
        )

        # ------------------------------------------------------
        # DEW CATEGORY
        # ------------------------------------------------------

        if (
            humidity >= 80
            and depression <= 4
        ):

            likelihood = "High"

        elif (
            humidity >= 70
            and depression <= 6
        ):

            likelihood = "Medium"

        else:

            likelihood = "Low"

        df.at[
            idx,
            "dew_likelihood"
        ] = likelihood

    if (idx + 1) % 100 == 0:

        print(
            f"Processed {idx + 1}/{len(df)}"
        )


# ==============================================================
# 18. WEATHER MATCH SUMMARY
# ==============================================================

weather_missing = (
    df["weather_start_temperature_2m"]
    .isna()
    .sum()
)

print("\n" + "=" * 70)
print("WEATHER MATCHING SUMMARY")
print("=" * 70)

print("Total matches:", len(df))

print(
    "Weather matched at start:",
    weather_start_matches
)

print(
    "Weather matched at +2 hours:",
    weather_2h_matches
)

print(
    "Weather missing:",
    weather_missing
)

print(
    "Weather match rate:",
    round(
        weather_start_matches / len(df) * 100,
        2
    ),
    "%"
)


# ==============================================================
# 19. DEW DISTRIBUTION
# ==============================================================

print("\n" + "=" * 70)
print("DEW LIKELIHOOD DISTRIBUTION")
print("=" * 70)

print(
    df["dew_likelihood"]
    .value_counts(dropna=False)
)


# ==============================================================
# 20. WEATHER BY MATCH TIMING
# ==============================================================

weather_timing = (
    df[
        df["weather_start_temperature_2m"]
        .notna()
    ]
    .groupby("match_timing")
    .agg(

        Matches=(
            "match_id",
            "count"
        ),

        Avg_Temperature=(
            "weather_start_temperature_2m",
            "mean"
        ),

        Avg_Humidity=(
            "weather_start_relative_humidity_2m",
            "mean"
        ),

        Avg_Dew_Point=(
            "weather_start_dew_point_2m",
            "mean"
        ),

        Avg_Dew_Depression=(
            "dew_point_depression_2h",
            "mean"
        ),

        Avg_Dew_Score=(
            "dew_proxy_score",
            "mean"
        ),
    )
)

print("\n" + "=" * 70)
print("WEATHER BY MATCH TIMING")
print("=" * 70)

print(
    weather_timing.round(2)
)


# ==============================================================
# 21. DEW × TOSS DECISION
# ==============================================================

analysis_df = df[
    df["dew_likelihood"].notna()
    & df["toss_decision"].notna()
].copy()


def calculate_summary(group):

    matches = len(group)

    toss_winner_win_pct = (
        group["toss_winner_won"]
        .mean()
        * 100
    )

    chase_win_pct = (
        group["chase_won"]
        .mean()
        * 100
    )

    bat_first_win_pct = (
        group["bat_first_won"]
        .mean()
        * 100
    )

    return pd.Series({

        "Matches": matches,

        "Toss_Winner_Win_Pct":
            round(
                toss_winner_win_pct,
                2
            ),

        "Chase_Win_Pct":
            round(
                chase_win_pct,
                2
            ),

        "Bat_First_Win_Pct":
            round(
                bat_first_win_pct,
                2
            ),
    })


dew_toss_analysis = (
    analysis_df
    .groupby(
        [
            "dew_likelihood",
            "toss_decision"
        ]
    )
    .apply(
        calculate_summary,
        include_groups=False
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("DEW LIKELIHOOD × TOSS DECISION")
print("=" * 70)

print(dew_toss_analysis)


# ==============================================================
# 22. DEW × MATCH OUTCOME
# ==============================================================

dew_outcome_analysis = (
    analysis_df
    .groupby("dew_likelihood")
    .apply(
        calculate_summary,
        include_groups=False
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("DEW LIKELIHOOD × MATCH OUTCOME")
print("=" * 70)

print(dew_outcome_analysis)


# ==============================================================
# 23. DEW × TIMING × TOSS
# ==============================================================

dew_timing_toss_analysis = (
    analysis_df
    .groupby(
        [
            "dew_likelihood",
            "match_timing",
            "toss_decision"
        ]
    )
    .apply(
        calculate_summary,
        include_groups=False
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("DEW × MATCH TIMING × TOSS DECISION")
print("=" * 70)

print(
    dew_timing_toss_analysis
)


# ==============================================================
# 24. SAVE FINAL DATASET
# ==============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ==============================================================
# 25. SAVE ANALYSIS FILES
# ==============================================================

weather_timing.to_csv(
    OUTPUT_DIR / "weather_by_timing.csv",
    index=False
)

dew_toss_analysis.to_csv(
    OUTPUT_DIR / "dew_toss_analysis.csv",
    index=False
)

dew_outcome_analysis.to_csv(
    OUTPUT_DIR / "dew_outcome_analysis.csv",
    index=False
)

dew_timing_toss_analysis.to_csv(
    OUTPUT_DIR / "dew_timing_toss_analysis.csv",
    index=False
)


# ==============================================================
# 26. FINAL VALIDATION
# ==============================================================

print("\n" + "=" * 70)
print("FINAL FILE SAVED")
print("=" * 70)

print(
    OUTPUT_FILE
)

print(
    "\nFinal shape:",
    df.shape
)

print(
    "\nFinal columns:",
    len(df.columns)
)

print(
    "\nWeather columns added:"
)

for col in weather_columns:
    print(" -", col)


# ==============================================================
# 27. CRITICAL COORDINATE CHECK
# ==============================================================

print("\n" + "=" * 70)
print("CRITICAL COORDINATE CHECK")
print("=" * 70)

for city in [
    "Bengaluru",
    "New Chandigarh",
]:

    if city in coordinate_cache:

        print(
            city,
            "→",
            coordinate_cache[city]
        )

    else:

        print(
            city,
            "→ MISSING"
        )


print("\n" + "=" * 70)
print("WEATHER ENRICHMENT COMPLETE")
print("=" * 70)